# Value counts

<small>**NOTE:** This notebook was to explore missingness / uniques / categorical counts.</small>

Import needed libraries and data:

In [ ]:
from itertools import chain
import numpy as np
import pandas as pd
import seaborn as sns

sns.set_theme("paper", "whitegrid")

In [ ]:
ema_rwd = pd.read_excel('../../data/ema_rwd/ema_rwd.xlsx', index_col=0).set_index("eu_pas_register_number")

Ensure that we loaded the full data (no cancelled excluded yet).

In [ ]:
len(ema_rwd)

## Missing / Unique counts

In [ ]:
original_name_mapping = {
    "Title": "title",
    "URL LINK": "url",
    "First published": "registration_date",
    "Updated": "update_date",
    "PURI": "puri",
    "EU PAS number": "eu_pas_register_number",
    "Study countries": "countries",
    "Study description": "description",
    "Study status": "state",
    "Institution conducting the study": "lead_institution_encepp",
    "Institution conducting the study if not in the list": "lead_institution_not_encepp",
    "Additional institutions": "additional_institutions_encepp",
    "Additional institutions if not in the list": "additional_institutions_not_encepp",
    "Networks conducting the study": "networks_encepp",
    "Additional networks if not in the list": "networks_not_encepp",
    "Date when funding contract was signed (Planned)": "funding_contract_date_planed",
    "Date when funding contract was signed (Actual)": "funding_contract_date_actual",
    "Study start date (Planned)": "data_collection_date_planed",
    "Study start date (Actual)": "data_collection_date_actual",
    "Data analysis start date (Planned)": "data_analysis_date_planed",
    "Data analysis start date (Actual)": "data_analysis_date_actual",
    "Date of interim report, if expected (Planned)": "iterim_report_date_planed",
    "Date of interim report, if expected (Actual)": "iterim_report_date_actual",
    "Date of final study report (Planned)": "final_report_date_planed",
    "Date of final study report (Actual)": "final_report_date_actual",
    "Source of funding": "funding_sources",
    "More details on source of funding": "funding_details",
    "Protocol": "protocol_document_name",
    "Was the study required by a regulatory body?": "requested_by_regulator",
    "Is the study required by a Risk Management Plan (RMP)?": "risk_management_plan",
    "Regulatory procedure number": "regulatory_procedure_number",
    "Study topic": "study_topic",
    "Study topic, other": "study_topic_other",
    "Study type": "study_type",
    "If ‘Not applicable’, further details on the study type": "study_type_other",
    "Scope of the study": "non_interventional_scopes",
    "If ‘other’, further details on the scope of the study": "non_interventional_scopes_other",
    "Non-interventional study design": "non_interventional_study_design",
    "Non-interventional study design, other": "non_interventional_study_design_other",
    "Name of medicine": "substance_brand_name",
    "Name of medicine, other": "substance_brand_name_other",
    "Study drug International non-proprietary name (INN) or common name": "substance_inn",
    "Anatomical Therapeutic Chemical (ATC) code": "substance_atc",
    "Medicinal condition to be studied": "medical_conditions",
    "Additional medical condition(s)": "additional_medical_conditions",
    "Population age groups": "age_population",
    "Special population of interest": "special_population",
    "Special population of interest, other": "special_population_other",
    "Estimated number of subjects": "number_of_subjects",
    "Outcomes": "outcomes",
    "Results tables": "result_tables_name",
    "Study report": "result_document_name",
    "Study, other information": "other_documents_name",
    "Study publications": "references",
    "Data source(s) ": "data_sources_registered_with_encepp",
    "Other linked data sources ": "data_sources_not_registered_with_encepp",
    "Check conformance": "check_conformance",
    "Check completeness": "check_completeness",
    "Check stability": "check_stability",
    "Check logical consistency": "check_logical_consistency",
    "Data characterisation conducted": "conducted_data_characterisation",
}

In [ ]:
variables_dependencies = {
    "age_population": ["age_population"],
    "collaboration_with_research_network": ["networks_encepp", "networks_not_encepp"],
    "funding_sources_grouped": [
        "funding_sources",
        "funding_details"
    ],
    "has_outcomes": ["outcomes"],
    "multiple_funding_sources": [
        "funding_details",  # $MATCHED
        "funding_sources",
    ],
    "number_of_countries_grouped": ["countries"],
    "number_of_studies_funded_by_biggest_sponsor_quartiles": [
        "funding_details",  # $MATCHED
        "funding_sources",
    ],
    "number_of_subjects_grouped": ["number_of_subjects"],
    "planned_duration_quartiles": [
        "data_collection_date_planed",
        "final_report_date_planed",
    ],
    "requested_by_regulator": ["requested_by_regulator"],
    "risk_management_plan": ["risk_management_plan"],
    "studied_medical_conditions": [
        "medical_conditions",
        "additional_medical_conditions",
    ],
    "study_type": ["study_type"],
    "updated_state": [
        "state",
        "description",
        "data_analysis_date_actual",
        "final_report_date_actual",
        "protocol_document_name",
        "result_tables_name",
        "result_document_name",
        "other_documents_name",
    ],
    "uses_established_data_source": [
        "data_sources_registered_with_encepp",
        "data_sources_not_registered_with_encepp",
    ],
    # OUTCOME VARIABLES
    # 'has_protocol': [],
    # 'has_result': [],
    # OTHER VARIABLES
    # "non_interventional_scopes": ["non_interventional_scopes"],
    # "non_interventional_study_design": ["non_interventional_study_design"],
    # "registration_year": ["registration_date"],
    # "registration_year_grouped": ["registration_year"],
    # "special_population": ["special_population"],
    # "study_topic": ["study_topic"],
    # "study_topic_grouped": ["study_topic"],
    # HELPER VARIABLES
    # Logistic regression variable
    "data_collection_days_difference": [
        # "data_collection_date_actual_override",
        "data_collection_date_actual",
    ],
    # # Variables used to detect due protocol population
    # "data_collection_busdays_difference": [
    #     # "data_collection_date_actual_override",
    #     "data_collection_date_actual",
    # ],
    # "due_protocol": [
    #     # "data_collection_date_actual_override",
    #     "data_collection_date_actual"
    # ],
    # # Variable used for other analysis
    # "due_protocol_year": [
    #     # "data_collection_date_actual_override",
    #     "data_collection_date_actual",
    # ],
    # Logistic regression variable
    "final_report_days_difference": [
        # "final_report_date_actual_override",
        "final_report_date_actual",
    ],
    # # Variables used to detect due result population
    # "final_report_busdays_difference": [
    #     # "final_report_date_actual_override",
    #     "final_report_date_actual",
    # ],
    # "due_result": [
    #     # "final_report_date_actual_override",
    #     "final_report_date_actual"
    # ],
    # # Variable used for other analysis
    # "due_result_year": [
    #     # "final_report_date_actual_override",
    #     "final_report_date_actual"
    # ],
}

In [ ]:
combination_categorical_cols = [
    "countries",
    "funding_sources",
    "study_topic",
    "non_interventional_scopes",
    "non_interventional_study_design",
    "age_population",
    "special_population",
]

single_categorical_cols = [
    "state",
    "requested_by_regulator",
    "risk_management_plan",
    "study_type",
    "check_conformance",
    "check_completeness",
    "check_stability",
    "check_logical_consistency",
    "conducted_data_characterisation",
]
 
categorical_cols = [
    *single_categorical_cols, 
    *combination_categorical_cols,
]

In [ ]:
info = ema_rwd.dtypes.rename("dtypes").rename_axis("columns").to_frame().join(
    ema_rwd.isna().sum().rename("null_count")
).join(
    ema_rwd.nunique().rename("nunique")
)

info

In [ ]:
dependency_df = pd.DataFrame.from_records(
    list(variables_dependencies.items()), columns=["variable", "dependency"]
).explode("dependency").set_index("dependency")

In [ ]:
info_variable = (
    info.assign(
        null_count=lambda df: df["null_count"].astype(str)
        + " ("
        + (100 * df["null_count"] / len(ema_rwd)).apply(lambda x: f"{x:.1f}")
        + ")",
        nunique=lambda df: df["nunique"].astype(str)
        + " ("
        + (100 * df["nunique"] / len(ema_rwd)).apply(lambda x: f"{x:.1f}")
        + ")",
    )
    .loc[sorted(list(set(list(chain.from_iterable(variables_dependencies.values())))))]
    .assign(
        dtypes=lambda df: np.where(
            df.index.to_series().isin(single_categorical_cols),
            "Categorical",
            np.where(
                df.index.to_series().isin(combination_categorical_cols),
                "Categorical (multiple can apply)",
                df["dtypes"].replace(
                    {"object": "String", "datetime64[ns]": "Date", "int64": "Integer"}
                ),
            ),
        )
    )
    .reset_index()
    .assign(
        original_fields=lambda df: df["columns"].replace(
            {v: k for k, v in original_name_mapping.items()}
        )
    )
    .set_index("columns")
    .rename(
        {
            "original_fields": "Metadata fields",
            "dtypes": "Metadata type",
            "null_count": "Missing count",
            "nunique": "Unique value count",
        },
        axis="columns",
    )
)

info_variable

In [ ]:
# Variables will be renamed by the values of this dict and sorted to match the order of the keys 
variable_rename_and_reorder_map = {
    # Requirements
    'risk_management_plan': 'Study required by a Risk Management Plan',
    'requested_by_regulator': 'Study required by a regulatory body',
    # Funding
    'funding_sources_grouped': 'Type of funding source',
    'multiple_funding_sources': 'Multiple funding sources',
    'number_of_studies_funded_by_biggest_sponsor_quartiles': 'Number of PAS funded by sponsor, quartiles',
    # Study
    'number_of_countries_grouped': 'Countries in which study is conducted',
    'study_type': 'Study type',
    'number_of_subjects_grouped': 'Estimated study population size',
    'age_population': 'Age of study population',
    'studied_medical_conditions': 'Medical condition(s) to be studied',
    'has_outcomes': 'Outcomes specified',
    # Research network and Data sources
    'collaboration_with_research_network': 'Collaboration of a research network',
    'uses_established_data_source': 'Study uses established data source',
    # Date based
    'updated_state': 'Status of study',
    'planned_duration_quartiles': 'Planned duration of study, quartiles',
    # Outcomes
    'has_protocol': 'Study made protocol public',
    'has_result': 'Study made final report public'
}

In [ ]:
missing_unique_table = (
    dependency_df.merge(
        info_variable,
        left_index=True,
        right_index=True,
        how="inner",
    )
    .sort_values("Metadata fields")
    .set_index("variable")
    .sort_index(
        level=0,
        key=lambda x: x.map(
            dict(
                zip(
                    variable_rename_and_reorder_map.keys(),
                    range(len(variable_rename_and_reorder_map)),
                )
            )
        ),
        sort_remaining=True,
    )
    .rename(index=variable_rename_and_reorder_map, level=0)
    .rename_axis("Variable")
)

md_fields = missing_unique_table.pop("Metadata fields")
missing_unique_table.insert(0, "Metadata fields", md_fields)

missing_unique_table

In [ ]:
missing_unique_table.to_excel("ema_rwd_final_statistics_tables_missings_uniques.xlsx")

## Categorical columns

### All
Uncomment the code to check the combination (single or multiple) for all categorical columns

In [ ]:
# for col in categorical_cols:
#     print(col)
#     display(
#         ema_rwd[col].value_counts(sort=True).to_frame()
#     )

In [ ]:
# for col in combination_categorical_cols:
#     print(f'Single component of {col}')
#     display(
#         ema_rwd[col].str.split('; ').explode().value_counts(sort=True).to_frame()
#     )

### Risk management plan

In [ ]:
ema_rwd.loc[ema_rwd["state"] != 'Planned', "risk_management_plan"].value_counts().to_frame()

In [ ]:
ema_rwd.loc[ema_rwd["state"] == 'Planned', "risk_management_plan"].value_counts().to_frame()

### Special Population

In [ ]:
display(
    ema_rwd["special_population"]
    .str.split("; ")
    .explode()
    .value_counts(sort=False)
    .sort_index()
    .rename_axis("Special Population")
    .rename("N")
    .to_frame()
    .assign(
        pct=lambda df: round(100 * df["N"] / len(ema_rwd), 1)
    )
    .rename(columns={
        'pct': '%'
    })
)

## Experiments

### References

In [ ]:
ema_rwd["references"].dropna().str.split("; ").explode().str.extract(
    r"https?://(?P<first>.*?)/"
)["first"].value_counts()

### State changes

In [ ]:
variables = pd.read_excel(
    '../../output/ema_rwd/ema_rwd_final_statistics_variables.xlsx', 
    sheet_name='all'
)

Simple state changes.

In [ ]:
states = variables[['state', 'updated_state']].fillna('Unspecified').value_counts(dropna=False, sort=False).to_frame().reset_index()

# states[states['state'] != states['updated_state']].to_clipboard(excel=True, sep='\t', index=False)
states[states['state'] != states['updated_state']]

Detailed state changes.

In [ ]:
states = (
    pd.read_excel("../../output/ema_rwd/ema_rwd_final.xlsx", index_col=0)[
        [
            "Eu Pas Register Number",
            "State",
            "$UPDATED_state",
            "$UPDATED_state_override",
            "$CANCELLED_MANUAL",
            "$CANCELLED_MANUAL_details",
        ]
    ]
    .set_index("Eu Pas Register Number")
    .assign(
        state=lambda df: df["State"].fillna("Unspecified"),
        update_based_on_dates_and_description=lambda df: np.where(
            df["$CANCELLED_MANUAL_details"].eq("based on description"),
            "Cancelled",
            df["$UPDATED_state"],
        ),
        update_based_on_dates_description_and_documents_state=lambda df: np.where(
            df["$CANCELLED_MANUAL"].eq(1),
            "Cancelled",
            df["$UPDATED_state_override"].combine_first(df["$UPDATED_state"]),
        ),
    )[
        [
            "state",
            "update_based_on_dates_and_description",
            "update_based_on_dates_description_and_documents_state",
        ]
    ]
    .value_counts(dropna=False, sort=False)
    .to_frame()
    .reset_index()
)

# states[states['state'] != states['update_based_on_dates_description_and_documents_state']].to_clipboard(excel=True, sep='\t', index=False)
states[
    states["state"] != states["update_based_on_dates_description_and_documents_state"]
]